# Day 3 — Advanced Pandas: Data Cleaning & Transformation
**Date:** 22 March 2026 | **Phase 1 — Week 1** | **Roadmap: DS-AI-75D**

> Mastering apply, map, missing data handling, pivot tables, and datetime operations.

In [1]:
print("Kernel is alive!")

Kernel is alive!


In [2]:
import pandas as pd

print(pd.__version__)

3.0.1


In [3]:
import pandas as pd
import numpy as np

# Load from local file
df = pd.read_csv(r"C:\DS-AI-75d\titanic.csv")

print(f"Shape: {df.shape}")
print(f"\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Shape: (891, 12)

Missing values:
Age         177
Cabin       687
Embarked      2
dtype: int64


## 2. apply & map — Custom Transformations

### 2.1 map — Transform a Single Column

In [4]:
# map — applies a dictionary to every value in a Series
df["Sex_encoded"] = df["Sex"].map({"male": 0, "female": 1})
print("Sex encoding:")
print(df[["Sex", "Sex_encoded"]].head(6))

Sex encoding:
      Sex  Sex_encoded
0    male            0
1  female            1
2  female            1
3  female            1
4    male            0
5    male            0


### 2.2 apply — Apply a Custom Function

In [5]:
# apply on a Series — create age groups
df["Age_group"] = df["Age"].apply(
    lambda x: (
        "Child"
        if x < 12
        else (
            "Teen"
            if x < 18
            else "Adult" if x < 60 else "Senior" if pd.notna(x) else "Unknown"
        )
    )
)

print("Age group distribution:")
print(df["Age_group"].value_counts())

Age group distribution:
Age_group
Adult      575
Unknown    177
Child       68
Teen        45
Senior      26
Name: count, dtype: int64


### 2.3 apply with a Custom Function — Extract Title from Name

In [6]:
# Extract title from passenger name using a custom function
def get_title(name):
    if "Mr." in name:
        return "Mr"
    elif "Mrs." in name:
        return "Mrs"
    elif "Miss." in name:
        return "Miss"
    elif "Master." in name:
        return "Master"
    else:
        return "Other"


df["Title"] = df["Name"].apply(get_title)

print("Title distribution:")
print(df["Title"].value_counts())

print("\nSurvival rate by title:")
print(df.groupby("Title")["Survived"].mean().round(3))

Title distribution:
Title
Mr        517
Miss      182
Mrs       125
Master     40
Other      27
Name: count, dtype: int64

Survival rate by title:
Title
Master    0.575
Miss      0.698
Mr        0.157
Mrs       0.792
Other     0.444
Name: Survived, dtype: float64


### 2.4 apply with axis=1 — Access Multiple Columns Per Row

In [7]:
# axis=1 means apply function row by row
df["FamilySize"] = df.apply(lambda row: row["SibSp"] + row["Parch"] + 1, axis=1)

df["FamilyType"] = df["FamilySize"].apply(
    lambda x: "Solo" if x == 1 else "Small" if x <= 4 else "Large"
)

print("Family type distribution:")
print(df["FamilyType"].value_counts())

print("\nSurvival rate by family type:")
print(df.groupby("FamilyType")["Survived"].mean().round(3))

Family type distribution:
FamilyType
Solo     537
Small    292
Large     62
Name: count, dtype: int64

Survival rate by family type:
FamilyType
Large    0.161
Small    0.579
Solo     0.304
Name: Survived, dtype: float64


## 3. Handling Missing Data
The most critical data cleaning skill. Real datasets are always messy.

### 3.1 Detecting Missing Data

In [8]:
# Full missing data report
missing = pd.DataFrame(
    {
        "Count": df.isnull().sum(),
        "Percent": (df.isnull().sum() / len(df) * 100).round(2),
    }
)
missing = missing[missing["Count"] > 0].sort_values("Percent", ascending=False)

print("Missing Data Report:")
print(missing)

print("\nVisual:")
for col, row in missing.iterrows():
    bar = "█" * int(row["Percent"] / 2)
    print(f"{col:12} {bar} {row['Percent']:.1f}%")

Missing Data Report:
          Count  Percent
Cabin       687    77.10
Age         177    19.87
Embarked      2     0.22

Visual:
Cabin        ██████████████████████████████████████ 77.1%
Age          █████████ 19.9%
Embarked      0.2%


### 3.2 Dropping Missing Data

In [9]:
print(f"Original shape: {df.shape}")

# Drop rows where ANY column has missing value
df_dropped_any = df.dropna()
print(f"After dropna():           {df_dropped_any.shape}")

# Drop rows where SPECIFIC column has missing value
df_dropped_age = df.dropna(subset=["Age"])
print(f"After dropna(Age only):   {df_dropped_age.shape}")

# Drop columns with more than 50% missing
threshold = len(df) * 0.5
df_dropped_cols = df.dropna(axis=1, thresh=threshold)
print(f"After dropping sparse cols: {df_dropped_cols.shape}")
print(f"Remaining columns: {df_dropped_cols.columns.tolist()}")

Original shape: (891, 17)
After dropna():           (183, 17)
After dropna(Age only):   (714, 17)
After dropping sparse cols: (891, 16)
Remaining columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Embarked', 'Sex_encoded', 'Age_group', 'Title', 'FamilySize', 'FamilyType']


### 3.3 Filling Missing Data — The Right Way

In [10]:
# Always work on a copy — never modify original
df_clean = df.copy()

# Strategy 1: Median for Age (robust to outliers)
age_median = df_clean["Age"].median()
df_clean["Age"] = df_clean["Age"].fillna(age_median)
print(f"Age missing after fill:      {df_clean['Age'].isnull().sum()}")

# Strategy 2: Mode for Embarked (categorical)
embarked_mode = df_clean["Embarked"].mode()[0]
df_clean["Embarked"] = df_clean["Embarked"].fillna(embarked_mode)
print(f"Embarked missing after fill: {df_clean['Embarked'].isnull().sum()}")

# Strategy 3: Constant for Cabin (too sparse)
df_clean["Cabin"] = df_clean["Cabin"].fillna("Unknown")
print(f"Cabin missing after fill:    {df_clean['Cabin'].isnull().sum()}")

print(f"\nTotal missing after cleaning: {df_clean.isnull().sum().sum()}")

Age missing after fill:      0
Embarked missing after fill: 0
Cabin missing after fill:    0

Total missing after cleaning: 0


### 3.4 Smart Imputation — Group-Based Filling

In [11]:
# Smarter approach — fill Age using median per Title
# A Mr. gets the median Mr. age, not the global median
df_smart = df.copy()

title_age_median = df_smart.groupby("Title")["Age"].median()
print("Median age by title:")
print(title_age_median)


def fill_age(row):
    if pd.isna(row["Age"]):
        return title_age_median[row["Title"]]
    return row["Age"]


df_smart["Age"] = df_smart.apply(fill_age, axis=1)
print(f"\nAge missing after smart fill: {df_smart['Age'].isnull().sum()}")
print("Much more accurate than global median!")

Median age by title:
Title
Master     3.5
Miss      21.0
Mr        30.0
Mrs       35.0
Other     44.5
Name: Age, dtype: float64

Age missing after smart fill: 0
Much more accurate than global median!


## 4. Pivot Tables & Crosstab
Excel-style analysis directly in Python.

### 4.1 Pivot Table — Survival by Class and Gender

In [12]:
pivot = pd.pivot_table(
    df, values="Survived", index="Pclass", columns="Sex", aggfunc="mean"
)
print("Survival rate by Class and Gender:")
print(pivot.round(3))

Survival rate by Class and Gender:
Sex     female   male
Pclass               
1        0.968  0.369
2        0.921  0.157
3        0.500  0.135


### 4.2 Multi-Metric Pivot Table

In [13]:
pivot_multi = pd.pivot_table(
    df,
    values=["Survived", "Fare", "Age"],
    index="Pclass",
    aggfunc={"Survived": "mean", "Fare": "mean", "Age": "median"},
)
print("Multi-metric pivot by class:")
print(pivot_multi.round(2))

Multi-metric pivot by class:
         Age   Fare  Survived
Pclass                       
1       37.0  84.15      0.63
2       29.0  20.66      0.47
3       24.0  13.68      0.24


### 4.3 Crosstab — Count Passengers by Category

In [14]:
ct = pd.crosstab(
    df["Pclass"],
    df["Survived"],
    rownames=["Class"],
    colnames=["Survived"],
    margins=True,
)
ct.columns = ["Died", "Survived", "Total"]
print("Passenger counts by class and survival:")
print(ct)

Passenger counts by class and survival:
       Died  Survived  Total
Class                       
1        80       136    216
2        97        87    184
3       372       119    491
All     549       342    891


## 5. Datetime Operations
Working with dates and times — essential for time series and business analysis.

### 5.1 Creating and Converting Dates

In [15]:
# Create a sample dataset with dates
data = {
    "PassengerId": [1, 2, 3, 4, 5],
    "Name": ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "BookingDate": [
        "1912-03-15",
        "1912-03-22",
        "1912-04-01",
        "1912-04-05",
        "1912-04-10",
    ],
    "Fare": [72.5, 50.0, 7.25, 30.0, 15.75],
}

df_dates = pd.DataFrame(data)

# Convert string to datetime — always do this
df_dates["BookingDate"] = pd.to_datetime(df_dates["BookingDate"])

print(df_dates.dtypes)
print(df_dates)

PassengerId             int64
Name                      str
BookingDate    datetime64[us]
Fare                  float64
dtype: object
   PassengerId   Name BookingDate   Fare
0            1  Alice  1912-03-15  72.50
1            2    Bob  1912-03-22  50.00
2            3  Carol  1912-04-01   7.25
3            4   Dave  1912-04-05  30.00
4            5    Eve  1912-04-10  15.75


### 5.2 Extracting Date Components

In [16]:
df_dates["Year"] = df_dates["BookingDate"].dt.year
df_dates["Month"] = df_dates["BookingDate"].dt.month
df_dates["Day"] = df_dates["BookingDate"].dt.day
df_dates["DayOfWeek"] = df_dates["BookingDate"].dt.day_name()

print("Date components extracted:")
print(df_dates[["Name", "BookingDate", "Year", "Month", "Day", "DayOfWeek"]])

Date components extracted:
    Name BookingDate  Year  Month  Day  DayOfWeek
0  Alice  1912-03-15  1912      3   15     Friday
1    Bob  1912-03-22  1912      3   22     Friday
2  Carol  1912-04-01  1912      4    1     Monday
3   Dave  1912-04-05  1912      4    5     Friday
4    Eve  1912-04-10  1912      4   10  Wednesday


### 5.3 Date Arithmetic

In [17]:
# Calculate days before Titanic departure
titanic_departure = pd.Timestamp("1912-04-10")

df_dates["DaysBeforeDeparture"] = (titanic_departure - df_dates["BookingDate"]).dt.days

print("Days before departure each passenger booked:")
print(df_dates[["Name", "BookingDate", "DaysBeforeDeparture"]])

# Filter by date
early_bookers = df_dates[df_dates["BookingDate"] < "1912-04-01"]
print(f"\nEarly bookers (before April): {len(early_bookers)}")
print(early_bookers[["Name", "BookingDate"]])

Days before departure each passenger booked:
    Name BookingDate  DaysBeforeDeparture
0  Alice  1912-03-15                   26
1    Bob  1912-03-22                   19
2  Carol  1912-04-01                    9
3   Dave  1912-04-05                    5
4    Eve  1912-04-10                    0

Early bookers (before April): 2
    Name BookingDate
0  Alice  1912-03-15
1    Bob  1912-03-22


## 6. Key Insights

1. **Title extraction** — Miss survived 70%, Mr only 16%
2. **Solo travellers** survived less than small family groups
3. **Group-based imputation** more accurate than global median
4. **Cabin column** — 77% missing, filled with 'Unknown'
5. **1st class women** — 97% survival, highest of any group
6. **3rd class men** — 13.5% survival, lowest of any group